**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../../docs/METHODS.md) for the implemented protocol.


In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import (
    build_knn_graph,
    spectral_gap_from_W,
    diffusion_map_embedding,
    spectral_clustering,
)
from utils.evaluation import plot_spectral_gap, plot_silhouette_vs_k
from v2.utils.runs import make_run_dir

LEVEL_DIR = PROJECT_ROOT / "v2" / "Level3"
DATA_DIR = LEVEL_DIR / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Per-run output folders. A single notebook execution creates one runs/runN
# that holds figures + results from Fix 3 + Fix 4. Edit the notebook
# between runs to compare configurations.
RUN_DIR = make_run_dir(LEVEL_DIR, notebook_label="level3_clustering_v2.ipynb")
FIG_DIR = RUN_DIR / "figures"
RES_DIR = RUN_DIR / "results"
print(f"this run -> {RUN_DIR}")

V1_FEAT = PROJECT_ROOT / "Level3" / "Data" / "level3_features.npz"
V1_SPEC = PROJECT_ROOT / "Level3" / "Data" / "level3_spectrograms.npz"


## Load original Level 3 data


In [ ]:
data = np.load(V1_FEAT, allow_pickle=True)
X_full = data["X"].astype(np.float32)
clip_ids_full = data["clip_ids"].astype(np.int64)

spec = np.load(V1_SPEC, allow_pickle=True)
S_full = spec["S"].astype(np.float32)

print(f"V1 (uncapped): {X_full.shape[0]} windows from {len(np.unique(clip_ids_full))} clips")


## Show the imbalance and cap at 3 windows / clip


In [ ]:
uniq_clips, counts = np.unique(clip_ids_full, return_counts=True)
order = np.argsort(-counts)
print("Top 10 clips by window count (uncapped):")
for cid, n in zip(uniq_clips[order][:10], counts[order][:10]):
    print(f"  clip {cid}: {n} windows")
print(f"Top 3 clips contribute {counts[order][:3].sum()} windows "
      f"({counts[order][:3].sum() / counts.sum() * 100:.1f}% of pool)")

MAX_WINDOWS_PER_CLIP = 3
RNG_CAP = np.random.default_rng(42)

keep_idx_list = []
for cid in uniq_clips:
    rows = np.where(clip_ids_full == cid)[0]
    if rows.size <= MAX_WINDOWS_PER_CLIP:
        keep_idx_list.extend(rows.tolist())
    else:
        pick = RNG_CAP.choice(rows, size=MAX_WINDOWS_PER_CLIP, replace=False)
        keep_idx_list.extend(sorted(pick.tolist()))

keep_idx = np.array(sorted(keep_idx_list), dtype=np.int64)
print(f"\nAfter capping at {MAX_WINDOWS_PER_CLIP}/clip: {keep_idx.size} windows "
      f"(was {X_full.shape[0]})")

X = X_full[keep_idx].copy()
clip_ids = clip_ids_full[keep_idx].copy()
S = S_full[keep_idx].copy()

# Save the capped pool so Fix 4 (later in this notebook) can reuse it.
# Snapshot inside the run folder + cache to Data/ for downstream loads.
for parent in (RES_DIR, DATA_DIR):
    np.savez_compressed(
        parent / "level3_features_v2_capped.npz",
        X=X, clip_ids=clip_ids, keep_idx=keep_idx,
    )
    np.savez_compressed(
        parent / "level3_spectrograms_v2_capped.npz",
        S=S, clip_ids=clip_ids, keep_idx=keep_idx,
    )
print("wrote capped .npz files to", RES_DIR)


## Standardize and rebuild the k-NN graph (k=7, σ=median, same as V1)


In [ ]:
scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)

K_GRAPH = 7
W = build_knn_graph(Xz, k=K_GRAPH, sigma="median")
print(f"W: {W.shape}, nnz={W.nnz}")


## Spectral gap on the capped graph


In [ ]:
eigs = spectral_gap_from_W(W, n_eigenvalues=10)
print("first 10 eigenvalues of L_sym (capped):", eigs)
plot_spectral_gap(eigs, FIG_DIR / "fix3_spectral_gap_v2.png",
                  title="Level 3 V2 (capped) — spectral gap")


## Spectral clustering for k=2..8 + silhouette scores


In [ ]:
K_RANGE = list(range(2, 9))
silhouettes = []
clusterings = {}
for k in K_RANGE:
    cl = spectral_clustering(W, k=k, random_state=0)
    clusterings[k] = cl
    s = silhouette_score(Xz, cl) if len(set(cl)) > 1 else float("nan")
    silhouettes.append(s)
    print(f"  k={k}: silhouette = {s:.4f}")

plot_silhouette_vs_k(silhouettes, K_RANGE, FIG_DIR / "fix3_silhouette_vs_k_v2.png")
best_k_v2 = K_RANGE[int(np.nanargmax(silhouettes))]
print(f"\nbest k (capped) = {best_k_v2}")


## Diffusion maps + 4-panel embedding figure (same panels as V1)


In [ ]:
diff_coords = diffusion_map_embedding(W, n_components=3, t=5)
tsne = TSNE(n_components=2, perplexity=min(30, max(5, X.shape[0] // 5)),
            init="pca", random_state=0)
tsne_coords = tsne.fit_transform(Xz)

cl_best = clusterings[best_k_v2]
cl_k3 = clusterings[3]
FLATNESS_MEAN_IDX = 13 + 13 + 3
flatness_mean = X[:, FLATNESS_MEAN_IDX]

import matplotlib as mpl
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
cmap_disc = mpl.colormaps.get_cmap("tab10").resampled(max(best_k_v2, 3))

ax = axes[0, 0]
for c in sorted(set(cl_best)):
    m = cl_best == c
    ax.scatter(diff_coords[m, 0], diff_coords[m, 1], s=20, alpha=0.7,
               color=cmap_disc(c), label=f"cluster {c}")
ax.set_title(f"Diffusion map t=5, colored by k*={best_k_v2} clusters (V2 capped)")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2"); ax.legend(fontsize=8)

ax = axes[0, 1]
sc = ax.scatter(diff_coords[:, 0], diff_coords[:, 1], c=flatness_mean,
                cmap="plasma", s=20, alpha=0.8)
ax.set_title("Diffusion map, colored by spectral flatness (V2 capped)")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="flatness")

ax = axes[1, 0]
for c in sorted(set(cl_best)):
    m = cl_best == c
    ax.scatter(tsne_coords[m, 0], tsne_coords[m, 1], s=20, alpha=0.7,
               color=cmap_disc(c), label=f"cluster {c}")
ax.set_title(f"t-SNE, colored by k*={best_k_v2} clusters (V2 capped)")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2"); ax.legend(fontsize=8)

ax = axes[1, 1]
cmap_3 = mpl.colormaps.get_cmap("tab10").resampled(3)
for c in sorted(set(cl_k3)):
    m = cl_k3 == c
    ax.scatter(diff_coords[m, 0], diff_coords[m, 1], s=20, alpha=0.7,
               color=cmap_3(c), label=f"cluster {c}")
ax.set_title("Diffusion map, k=3 (biological reference, V2 capped)")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2"); ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fix3_embeddings_v2.png", dpi=150)
plt.close(fig)
print("saved embeddings_v2.png")


## V1's circular flatness-tercile validation, recomputed on capped pool

Reported here only for backward compatibility with V1; Fix 4 adds the
non-circular proxy.


In [ ]:
q1, q2 = np.quantile(flatness_mean, [1/3, 2/3])
proxy_flat = np.zeros_like(flatness_mean, dtype=np.int64)
proxy_flat[flatness_mean > q1] = 1
proxy_flat[flatness_mean > q2] = 2

ari_k3_flat = float(adjusted_rand_score(proxy_flat, cl_k3))
nmi_k3_flat = float(normalized_mutual_info_score(proxy_flat, cl_k3))
ari_best_flat = float(adjusted_rand_score(proxy_flat, cl_best))
nmi_best_flat = float(normalized_mutual_info_score(proxy_flat, cl_best))
print(f"k=3   vs flatness proxy (V1, circular): ARI={ari_k3_flat:.4f}, NMI={nmi_k3_flat:.4f}")
print(f"k={best_k_v2} (best) vs flatness proxy (V1, circular): "
      f"ARI={ari_best_flat:.4f}, NMI={nmi_best_flat:.4f}")


## Autoencoder + k-means baseline on the capped pool


In [ ]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(46, 32), nn.ReLU(), nn.Linear(32, 16))
        self.dec = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 46))
    def forward(self, x):
        z = self.enc(x); return self.dec(z), z

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
model = AE().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
crit = nn.MSELoss()

Xt = torch.from_numpy(Xz).to(device)
ds = TensorDataset(Xt)
dl = DataLoader(ds, batch_size=32, shuffle=True)

EPOCHS = 200
losses = []
for ep in range(EPOCHS):
    model.train(); ep_loss = 0.0; n = 0
    for (xb,) in dl:
        opt.zero_grad()
        rec, _ = model(xb)
        loss = crit(rec, xb)
        loss.backward(); opt.step()
        ep_loss += loss.item() * xb.size(0); n += xb.size(0)
    losses.append(ep_loss / n)
print(f"AE final loss: {losses[-1]:.4f}")

model.eval()
with torch.no_grad():
    _, Z = model(Xt)
Z = Z.cpu().numpy()

ae_silh = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    lab = km.fit_predict(Z)
    s = silhouette_score(Z, lab) if len(set(lab)) > 1 else float("nan")
    ae_silh.append(s)
    print(f"  AE+kmeans k={k}: silhouette={s:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_RANGE, silhouettes, marker="o", label="spectral clustering (capped)")
ax.plot(K_RANGE, ae_silh, marker="s", label="AE + k-means (capped)")
ax.set_xlabel("k"); ax.set_ylabel("silhouette score")
ax.set_title("Level 3 V2 (capped) — silhouette: spectral vs AE")
ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fix3_silhouette_compare_v2.png", dpi=150)
plt.close(fig)


## Persist Fix 3 results


In [ ]:
results_fix3 = {
    "uncapped": {
        "n_windows": int(X_full.shape[0]),
        "n_clips": int(uniq_clips.size),
        "top3_clip_window_count": int(counts[order][:3].sum()),
        "top3_clip_window_pct": float(counts[order][:3].sum() / counts.sum()),
    },
    "capped": {
        "max_windows_per_clip": MAX_WINDOWS_PER_CLIP,
        "n_windows": int(X.shape[0]),
        "n_clips": int(uniq_clips.size),
    },
    "spectral_clustering": {
        "k_range": K_RANGE,
        "silhouette": [float(s) for s in silhouettes],
        "best_k": int(best_k_v2),
        "best_silhouette": float(max(silhouettes)),
    },
    "ae_kmeans": {
        "k_range": K_RANGE,
        "silhouette": [float(s) for s in ae_silh],
        "ae_final_loss": float(losses[-1]),
    },
    "v1_circular_flatness_proxy_for_reference": {
        "k3_ARI": ari_k3_flat,
        "k3_NMI": nmi_k3_flat,
        f"best_k{best_k_v2}_ARI": ari_best_flat,
        f"best_k{best_k_v2}_NMI": nmi_best_flat,
    },
}
out_json = RES_DIR / "level3_results_v2_fix3.json"
out_json.write_text(json.dumps(results_fix3, indent=2), encoding="utf-8")
print("wrote", out_json)

# Save clusterings for Fix 4 to consume — snapshot in run + cache in Data/.
clusters_payload = {f"cl_k{k}": clusterings[k] for k in K_RANGE}
clusters_payload["best_k"] = np.array(best_k_v2)
np.savez_compressed(RES_DIR / "level3_v2_clusterings.npz", **clusters_payload)
np.savez_compressed(DATA_DIR / "level3_v2_clusterings.npz", **clusters_payload)
print("wrote clusterings .npz")


# Level 3 V2 — Fix 4: non-circular validation proxy

V1 used spectral flatness as both a clustering feature *and* the
validation reference (flatness terciles → ARI/NMI). Fix 4 computes
validation features that the clusterer did **not** see, derived directly
from the log-Mel spectrogram rather than the 46-dim feature vector.

Three new validation-only features:

1. **Spectral entropy** — Shannon entropy of the per-frame Mel-energy
   distribution, averaged over frames. High = broadband (clicks),
   low = peaked (tonal whistles).
2. **Dominant-bin continuity** — variance of the per-frame argmax Mel bin
   (the peak frequency track). Smooth contours (whistles) → low variance;
   discontinuous tracks (clicks) → high variance.
3. **Frame-energy PAPR** — peak-over-average ratio of per-frame total
   energy. Impulsive sounds (clicks) → high PAPR; sustained sounds
   (whistles) → low PAPR.

We then cluster these 3 features with k-means (k=3) for an independent
proxy label set, and also bin PAPR into terciles as a simpler proxy. ARI
and NMI are computed for both proxies vs the spectral clustering
assignments from Fix 3.


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from v2.utils.runs import make_run_dir

LEVEL_DIR = PROJECT_ROOT / "v2" / "Level3"
DATA_DIR = LEVEL_DIR / "Data"

# Inherit RUN_DIR / FIG_DIR / RES_DIR from the Fix 3 block above when this
# script is part of the combined Level 3 notebook; otherwise create a
# standalone run.
try:
    RUN_DIR  # type: ignore[name-defined]
except NameError:
    RUN_DIR = make_run_dir(LEVEL_DIR, notebook_label="level3_clustering_v2.ipynb (fix4 standalone)")
    FIG_DIR = RUN_DIR / "figures"
    RES_DIR = RUN_DIR / "results"
    print(f"this run -> {RUN_DIR}")


## Load capped spectrograms and Fix 3 cluster assignments


In [ ]:
# `Data/...capped.npz` is updated by every run of the Fix 3 block (latest
# wins). When Fix 3 ran in this same notebook execution, this is the file
# we just wrote.
spec = np.load(DATA_DIR / "level3_spectrograms_v2_capped.npz", allow_pickle=True)
S = spec["S"].astype(np.float32)            # (N, 128, T) log-Mel
clip_ids = spec["clip_ids"].astype(np.int64)
print(f"capped spectrograms: {S.shape}")

clusterings = np.load(DATA_DIR / "level3_v2_clusterings.npz", allow_pickle=True)
K_RANGE = list(range(2, 9))
cl = {k: clusterings[f"cl_k{k}"] for k in K_RANGE}
best_k = int(clusterings["best_k"])
print(f"best k from Fix 3 = {best_k}")

# Capped feature pool — needed only for the V1-style flatness reference.
feat = np.load(DATA_DIR / "level3_features_v2_capped.npz", allow_pickle=True)
X = feat["X"].astype(np.float32)
flatness_mean = X[:, 13 + 13 + 3]  # 4th of the 4 spectral stats means


## Compute the 3 validation features from log-Mel spectrograms

The spectrograms are log-Mel (`log(S + 1e-9)`). For energy-based
computations we need linear power, so we exponentiate back: P = exp(logS).


In [ ]:
def spectral_entropy(P_lin: np.ndarray) -> float:
    """Frame-averaged Shannon entropy of the Mel-energy distribution.

    P_lin: (n_mels, T) linear-Mel power. Each frame's column is normalized
    to a probability distribution over mel bins, then we average the Shannon
    entropy across frames.
    """
    eps = 1e-12
    col_sums = P_lin.sum(axis=0, keepdims=True)
    col_sums = np.where(col_sums > 0, col_sums, eps)
    Pn = P_lin / col_sums
    H = -np.sum(Pn * np.log(Pn + eps), axis=0)  # per-frame entropy, shape (T,)
    return float(H.mean())


def dominant_bin_variance(P_lin: np.ndarray) -> float:
    """Variance of the per-frame argmax Mel bin (peak frequency track).

    Whistles have a smooth contour → low variance.
    Clicks scatter the peak across many bins → high variance.
    """
    if P_lin.shape[1] < 2:
        return 0.0
    peaks = np.argmax(P_lin, axis=0).astype(np.float64)
    return float(np.var(peaks))


def frame_energy_papr(P_lin: np.ndarray) -> float:
    """Peak-to-average ratio of per-frame total energy."""
    e = P_lin.sum(axis=0)
    if e.size == 0 or e.mean() <= 0:
        return 0.0
    return float(e.max() / e.mean())


N = S.shape[0]
val_feats = np.zeros((N, 3), dtype=np.float64)
for i in range(N):
    P_lin = np.exp(S[i].astype(np.float64))  # back to linear
    val_feats[i, 0] = spectral_entropy(P_lin)
    val_feats[i, 1] = dominant_bin_variance(P_lin)
    val_feats[i, 2] = frame_energy_papr(P_lin)

print("validation feature ranges:")
print(f"  spectral entropy:     {val_feats[:,0].min():.3f}–{val_feats[:,0].max():.3f}")
print(f"  dominant bin variance:{val_feats[:,1].min():.3f}–{val_feats[:,1].max():.3f}")
print(f"  frame-energy PAPR:    {val_feats[:,2].min():.3f}–{val_feats[:,2].max():.3f}")


## Independence check: correlation between val features and clustering features

These should NOT be perfectly correlated with the flatness column the V1
proxy used. We compute Pearson correlation with `flatness_mean` so we can
show the new proxy is genuinely a different signal.


In [ ]:
def corr(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a - a.mean(); b = b - b.mean()
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float((a @ b) / denom) if denom > 0 else 0.0

print("Correlation with V1's `flatness_mean` (for context - closer to 0 = more independent):")
for j, name in enumerate(["spectral_entropy", "dominant_bin_var", "papr"]):
    print(f"  corr(flatness, {name}) = {corr(flatness_mean, val_feats[:,j]):+.3f}")


## Build proxy label sets

Two proxies:
  - **k-means(3)** on standardized validation features → 3-class proxy.
  - **PAPR terciles** (low / mid / high) → 3-class proxy.


In [ ]:
val_z = StandardScaler().fit_transform(val_feats).astype(np.float64)
km = KMeans(n_clusters=3, n_init=20, random_state=0)
proxy_kmeans = km.fit_predict(val_z)

papr = val_feats[:, 2]
q1, q2 = np.quantile(papr, [1/3, 2/3])
proxy_papr = np.zeros_like(papr, dtype=np.int64)
proxy_papr[papr > q1] = 1
proxy_papr[papr > q2] = 2
print(f"proxy_kmeans class counts: {np.bincount(proxy_kmeans)}")
print(f"proxy_papr  class counts: {np.bincount(proxy_papr)}")


## V1's circular flatness-tercile proxy on the capped pool — for reference


In [ ]:
qf1, qf2 = np.quantile(flatness_mean, [1/3, 2/3])
proxy_flat = np.zeros_like(flatness_mean, dtype=np.int64)
proxy_flat[flatness_mean > qf1] = 1
proxy_flat[flatness_mean > qf2] = 2


## Compute ARI / NMI for each k-clustering against each proxy


In [ ]:
table = {}
for k in K_RANGE:
    preds = cl[k]
    table[k] = {
        "n_clusters": int(len(set(preds))),
        "vs_proxy_kmeans3_independent": {
            "ARI": float(adjusted_rand_score(proxy_kmeans, preds)),
            "NMI": float(normalized_mutual_info_score(proxy_kmeans, preds)),
        },
        "vs_proxy_papr_terciles_independent": {
            "ARI": float(adjusted_rand_score(proxy_papr, preds)),
            "NMI": float(normalized_mutual_info_score(proxy_papr, preds)),
        },
        "vs_proxy_flatness_terciles_v1_circular": {
            "ARI": float(adjusted_rand_score(proxy_flat, preds)),
            "NMI": float(normalized_mutual_info_score(proxy_flat, preds)),
        },
    }
    print(f"k={k}: "
          f"kmeans3 ARI={table[k]['vs_proxy_kmeans3_independent']['ARI']:.3f} "
          f"NMI={table[k]['vs_proxy_kmeans3_independent']['NMI']:.3f} | "
          f"PAPR terciles ARI={table[k]['vs_proxy_papr_terciles_independent']['ARI']:.3f} "
          f"NMI={table[k]['vs_proxy_papr_terciles_independent']['NMI']:.3f} | "
          f"flatness V1 ARI={table[k]['vs_proxy_flatness_terciles_v1_circular']['ARI']:.3f}")


## Headline comparison: V1 (circular) vs V2 (independent) at k=2 and k=3


In [ ]:
print("\n=== Validation proxy comparison ===")
for k in (2, 3, best_k):
    if k not in table:
        continue
    v1 = table[k]["vs_proxy_flatness_terciles_v1_circular"]
    v2k = table[k]["vs_proxy_kmeans3_independent"]
    v2p = table[k]["vs_proxy_papr_terciles_independent"]
    print(f"\nk={k}")
    print(f"  V1 flatness terciles (CIRCULAR):       ARI={v1['ARI']:.3f}  NMI={v1['NMI']:.3f}")
    print(f"  V2 k-means(3) on val feats (indep):    ARI={v2k['ARI']:.3f}  NMI={v2k['NMI']:.3f}")
    print(f"  V2 PAPR terciles (indep):              ARI={v2p['ARI']:.3f}  NMI={v2p['NMI']:.3f}")


## Diffusion map colored by independent proxy (sanity check)


In [ ]:
from utils.graph import build_knn_graph, diffusion_map_embedding

Xz = StandardScaler().fit_transform(X).astype(np.float32)
W = build_knn_graph(Xz, k=7, sigma="median")
diff = diffusion_map_embedding(W, n_components=2, t=5)

import matplotlib as mpl
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap3 = mpl.colormaps.get_cmap("tab10").resampled(3)

for ax, prx, title in [
    (axes[0], proxy_kmeans, "Proxy: k-means(3) on (entropy, dom-bin var, PAPR)"),
    (axes[1], proxy_papr, "Proxy: PAPR terciles (low/mid/high)"),
    (axes[2], proxy_flat, "Proxy V1: flatness terciles (CIRCULAR)"),
]:
    for c in sorted(set(prx)):
        m = prx == c
        ax.scatter(diff[m, 0], diff[m, 1], s=20, alpha=0.7, color=cmap3(c), label=f"{c}")
    ax.set_title(title); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fix4_proxy_compare.png", dpi=150)
plt.close(fig)
print("saved fix4_proxy_compare.png")


## Persist Fix 4 results


In [ ]:
results_fix4 = {
    "validation_features": ["spectral_entropy", "dominant_bin_variance", "frame_energy_papr"],
    "correlations_with_flatness_mean": {
        "spectral_entropy": corr(flatness_mean, val_feats[:, 0]),
        "dominant_bin_variance": corr(flatness_mean, val_feats[:, 1]),
        "papr": corr(flatness_mean, val_feats[:, 2]),
    },
    "proxy_class_counts": {
        "kmeans3_independent": np.bincount(proxy_kmeans).tolist(),
        "papr_terciles_independent": np.bincount(proxy_papr).tolist(),
        "flatness_terciles_v1_circular": np.bincount(proxy_flat).tolist(),
    },
    "per_k_metrics": table,
    "best_k_from_fix3": best_k,
}
out_json = RES_DIR / "level3_results_v2_fix4.json"
out_json.write_text(json.dumps(results_fix4, indent=2), encoding="utf-8")
print("wrote", out_json)

val_payload = dict(
    val_feats=val_feats,
    proxy_kmeans=proxy_kmeans,
    proxy_papr=proxy_papr,
    proxy_flat=proxy_flat,
)
np.savez_compressed(RES_DIR / "level3_v2_validation_features.npz", **val_payload)
np.savez_compressed(DATA_DIR / "level3_v2_validation_features.npz", **val_payload)
print("wrote validation features .npz")
